# SurvFace 02. Compressor fit

`training_manifest.csv`의 development 임베딩만 사용해 PCA와 direct-origin PQ를 학습합니다. 공식 gallery/registered/unmated test는 `fit_on_survface_official_test=False`인 평가 전용 데이터이며 이 단계에 입력되지 않습니다.

진행 로그는 장시간 DB scan과 모델 학습에서 약 10% 경계(10, 20, …, 100%)에만 출력합니다. 완료된 동일 phase가 있으면 checksum을 검증해 재사용합니다.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("프로젝트 루트(C:/ronbun)를 찾을 수 없습니다.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from research.database import create_database_engine, load_database_settings
from research.experiments import fit_survface_compressors
from research.runtime import ProgressReporter, RunStore, resolve_active_run

MODE = "real"
DATA_FRACTION = 1.0
SEED = 42
EXECUTE_STAGE = True
RUN_ROOT = PROJECT_ROOT / "runs" / "survface"
TRAINING_MANIFEST_PATH = PROJECT_ROOT / "data/interim/survface/training_manifest.csv"
RUN_DIR = resolve_active_run(
    RUN_ROOT,
    environment_variable="RONBUN_SURVFACE_RUN_DIR",
)
PROGRESS = ProgressReporter(
    "SurvFace 02 compressor fit",
    heartbeat_seconds=None,
    milestone_percent=10,
)
preflight = {
    "mode": MODE,
    "data_fraction": DATA_FRACTION,
    "seed": SEED,
    "execute_stage": EXECUTE_STAGE,
    "run_dir": str(RUN_DIR),
    "training_manifest": str(TRAINING_MANIFEST_PATH),
}
preflight

In [ ]:
result = {"status": "not_executed", **preflight}
if EXECUTE_STAGE:
    run = RunStore.open(RUN_DIR)
    config = run.config
    compression = config["compression"]
    pca_dimensions = tuple(int(value) for value in compression["pca"]["dimensions"])
    pq_settings = tuple(
        (int(item["m"]), int(item["nbits"]))
        for item in compression["pq"]["settings"]
    )
    training_manifest = pd.read_csv(TRAINING_MANIFEST_PATH)
    engine = create_database_engine(load_database_settings())
    bundle = fit_survface_compressors(
        run,
        engine,
        training_manifest=training_manifest,
        training_manifest_path=TRAINING_MANIFEST_PATH,
        project_root=PROJECT_ROOT,
        pca_dimensions=pca_dimensions,
        pq_settings=pq_settings,
        seed=SEED,
        batch_size=int(compression.get("batch_size", 1024)),
        progress=PROGRESS.callback(key_prefix=f"{run.run_id}:"),
    )
    result = {
        "status": "completed",
        "run_id": run.run_id,
        "attempt": f"A{bundle.attempt:03d}",
        "fit_count": bundle.fit_count,
        "pca_profiles": list(bundle.pcas),
        "pq_profiles": list(bundle.pqs),
        "fit_source": bundle.summary["fit_source"],
        "official_test_fit": bundle.summary["official_test_fit"],
    }
result

## 다음 단계

`fit_source=survface_training_development`, `official_test_fit=false`, PCA/PQ profile 수와 fit count를 확인한 뒤 `01_official_compressed_materialization_and_index.ipynb`로 이동합니다.